# LLMBuilder LoRA re-train — Kaggle T4 (free GPU)

Same Track A run as the proven Nov 10 Colab T4 run: QLoRA 4-bit NF4, r=64, alpha=128, lr 2e-4, 2 epochs, max_seq 2048, micro 1 × accum 32, **152 total steps**, corpus 2,426 rows (Lucid knowledge only — no external data, ever).

## The GPU pool — how this notebook fits

GPUs are interchangeable handoff workers in a pool, occupying open parallel slots. The CPU runs in parallel underneath holding the run up the whole time as the safety net — it does NOT occupy a GPU slot. The hierarchy (Gary, 2026-11-11):

1. **A GPU runs normally** (Colab T4 right now).
2. **If it dies**, the CPU is already holding the live run in parallel, and the next GPU worker (this notebook, on Kaggle) jumps into an open slot and takes the **live state straight from the CPU** — no checkpoint involved in that step.
3. **ONLY if worse comes to worst** — the live hold is gone too — does the **checkpoint** step in as the fallback. Checkpoints are the **last resort**, not the handoff. The `save_steps: 25` fix means those fallback checkpoints land at 25/50/75/100/125/150.

So this notebook has three modes, decided by the state-intake cell:

- **Live state (PRIMARY handoff):** a `live-state` bundle from the CPU hold (`live_state.json`). The GPU work continues from the CPU's live state at the exact step the GPU died. **Status: the intake is wired (this notebook detects + verifies the bundle), but the continue-from-live-state execution is NOT wired yet — the CPU live hold itself does not exist yet, so there is no live state to receive. If a live-state bundle is attached, this notebook stops and says so plainly instead of silently training fresh or falling back to a checkpoint.**
- **Checkpoint (LAST RESORT only):** a `checkpoint-<N>` bundle (`handoff.json`, kind `checkpoint`) — the latest disk snapshot, resumed with `--resume-from-checkpoint`. Used only when the live hold is gone too (or for a cold-start worker that was never in the parallel run). Never presented as the handoff.
- **Fresh start:** no state attached — the full 152-step run from step 0.

**Prerequisites (do these before Run All):**
1. **Dataset attached:** `llmbuilder-lora-run` (code + corpus) must be attached to this notebook — it is listed in `kernel-metadata.json` under `dataset_sources`.
2. **State bundle attached (only for a handoff):** for the GPU-death jump-in, attach the `llmbuilder-live-state` dataset (the CPU hold's live-state bundle, made with `tools/export_state.py --kind live-state`). For the last-resort fallback, attach the `llmbuilder-checkpoint` dataset instead. Attach at most one state bundle — if both are attached, the live state wins and the checkpoint is ignored.
3. **Secret `HF_TOKEN`:** right panel → Add-ons → Secrets → add a secret named `HF_TOKEN` whose value is the Hugging Face token for the account that accepted the Llama 3.2 1B license. The token is never written into any file; the notebook reads it from the secret and only ever prints the HF username.
4. **GPU on:** Settings → Accelerator → **GPU T4**. (Kaggle GPU access needs phone verification on the account.)
5. **Internet on:** needed to download the base weights from Hugging Face.

**Expected time:** setup ~5–10 min; a fresh full run ~67 min on a Kaggle T4 (measured 67 min on a Colab T4, same run). A live-state jump-in or a last-resort checkpoint resume runs only the remaining steps (e.g. from step 50: ~45 min). Kaggle sessions cap at 12 h — this run fits easily.


In [ ]:
import sys, platform
print("python:", sys.version)
print("platform:", platform.platform())
try:
    import torch
    print("torch:", torch.__version__, "| cuda available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0), "| capability:", torch.cuda.get_device_capability(0))
except ImportError as e:
    print("torch NOT installed yet (expected before the pip cell):", e)


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv


In [ ]:
import hashlib, json, shutil, subprocess, time
from pathlib import Path

def run_cmd(argv, label):
    """Run a command streaming its output live; fail loudly on non-zero exit."""
    print(f"\n===== {label} =====", flush=True)
    print(" ".join(str(a) for a in argv), flush=True)
    t0 = time.time()
    proc = subprocess.Popen([str(a) for a in argv], stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="", flush=True)
    rc = proc.wait()
    print(f"[{label}] exit {rc} after {(time.time() - t0) / 60:.1f} min", flush=True)
    assert rc == 0, f"{label} failed with exit code {rc}"
    return rc

# The attached dataset is mounted read-only under /kaggle/input. Find it by
# the trainer script (robust to the exact dataset slug), then copy it into
# /kaggle/working — the trainer writes checkpoints + manifest, so it needs a
# writable copy. The repo-relative layout is kept, so train_lora.py's
# REPO_ROOT (= parents[2] of scripts/training/train_lora.py) resolves.
cands = [p for p in Path("/kaggle/input").rglob("scripts/training/train_lora.py")]
assert len(cands) == 1, f"expected exactly 1 train_lora.py under /kaggle/input, found {len(cands)}"
SRC = cands[0].parents[2]  # dataset root
DST = Path("/kaggle/working/llmbuilder")
if DST.exists():
    shutil.rmtree(DST)
shutil.copytree(SRC, DST)
print("copied", SRC, "->", DST)

# Verify the corpus before anything trains: 2426 rows, sha256 from its manifest.
CORPUS = DST / "datasets/lucid_corpus/lucid_corpus.jsonl"
MANIFEST = json.loads((DST / "datasets/lucid_corpus/manifest.json").read_text())
sha = hashlib.sha256(CORPUS.read_bytes()).hexdigest()
rows = sum(1 for _ in CORPUS.open())
print("corpus sha256:", sha)
print("corpus rows:", rows, "| manifest rows:", MANIFEST["total_rows"])
assert sha == MANIFEST["corpus_sha256"] == "dc5cdeaf0890691e794a075a7a9bdb80a7e1a4460826fa48c8764f432b322cb8", \
    "corpus sha256 MISMATCH — do not train"
assert rows == 2426 == MANIFEST["total_rows"], "corpus row count MISMATCH — do not train"
print("expected total steps:", -(-2426 * 2 // 32), "(2426 rows x 2 epochs / 32 effective batch)")

# Verify the Kaggle config carries save_steps 25.
import yaml
cfg = yaml.safe_load((DST / "configs/training/sft_lora_kaggle.yaml").read_text())
t = cfg["trainer"]
print("config trainer:", {k: t[k] for k in ("rank", "alpha", "learning_rate", "epochs", "max_seq_length", "save_steps")})
assert t["save_steps"] == 25, "sft_lora_kaggle.yaml must set trainer.save_steps: 25"
assert (t["rank"], t["alpha"], t["learning_rate"], t["epochs"], t["max_seq_length"]) == (64, 128, 2e-4, 2, 2048)

TRAIN = DST / "scripts/training/train_lora.py"
DATA = CORPUS
CFG = DST / "configs/training/sft_lora_kaggle.yaml"
OUT = Path("/kaggle/working/run1")
print("TRAIN:", TRAIN)
print("DATA :", DATA)
print("CFG  :", CFG)
print("OUT  :", OUT)


In [ ]:
import sys
# Install the pinned training stack. torch is NOT pinned in
# requirements-lora.txt on purpose — Kaggle preinstalls the CUDA build and
# this must not downgrade/replace it (same rule as Colab).
run_cmd([sys.executable, "-m", "pip", "install", "-q", "-r", str(DST / "requirements-lora.txt")],
        "pip install requirements-lora.txt")
import torch, transformers, trl, peft, datasets, bitsandbytes
print("torch", torch.__version__, "| transformers", transformers.__version__,
      "| trl", trl.__version__, "| peft", peft.__version__,
      "| datasets", datasets.__version__, "| bitsandbytes", bitsandbytes.__version__)


In [ ]:
import os
# Hugging Face token comes from a Kaggle Secret named HF_TOKEN (notebook
# Settings -> Add-ons -> Secrets). It is never written into any file and
# never printed — only the HF username is shown. The trainer subprocess
# inherits os.environ, so HF_TOKEN reaches it.
try:
    from kaggle_secrets import UserSecretsClient
    _tok = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as e:
    raise RuntimeError(
        "Kaggle Secret HF_TOKEN is missing or unreadable. In the Kaggle notebook UI: "
        "right panel -> Add-ons -> Secrets -> add a secret named HF_TOKEN whose value is "
        "the Hugging Face token for the account that accepted the Llama 3.2 1B license. "
        f"Underlying error: {e}")
assert _tok, "HF_TOKEN secret is empty"
os.environ["HF_TOKEN"] = _tok
from huggingface_hub import HfApi
print("HF login OK as:", HfApi().whoami(token=_tok)["name"])
del _tok  # raw token stays only in os.environ for the trainer subprocess


In [ ]:
# Dry run: validates config + dataset and resolves the tokenizer only.
# No model weights, no GPU.
run_cmd([sys.executable, str(TRAIN), "--dataset", str(DATA), "--config", str(CFG),
         "--output-dir", str(OUT), "--save-steps", "25", "--dry-run"], "dry-run")


In [ ]:
# Smoke test: 1 training step on 8 rows — proves the GPU + stack work
# before the ~67 min full run. Skip this cell only if you are re-running
# after a known-good smoke test.
run_cmd([sys.executable, str(TRAIN), "--dataset", str(DATA), "--config", str(CFG),
         "--output-dir", str(OUT), "--save-steps", "25", "--smoke-test"],
        "smoke-test (1 step, 8 rows)")


In [ ]:
# STATE INTAKE — live state (PRIMARY) or checkpoint (LAST RESORT).
# This cell decides which mode the run takes. It scans /kaggle/input for a
# state bundle attached as a Kaggle dataset and verifies it BEFORE anything
# trains. A bundle that fails verification is discarded — never train from
# an unverified bundle.
#
# The two kinds (different mechanisms, do not mix them):
#  * live-state  — the CPU hold's LIVE state at GPU-death time
#    (live_state.json, schema llmbuilder-live-state/v1). PRIMARY handoff:
#    the next GPU worker jumps into an open slot and takes the live state
#    straight from the CPU. No checkpoint involved in that step.
#  * checkpoint  — a periodic disk snapshot, checkpoint-<N>
#    (handoff.json, kind checkpoint). LAST RESORT ONLY: used when the live
#    hold is gone too (or for a cold-start worker that was never in the
#    parallel run). It is NOT the handoff.
# A bundle is either a zip (llmbuilder-live-state-<N>.zip /
# llmbuilder-checkpoint-<N>.zip) or an already-extracted dir. Both are
# accepted. Sets LIVE_STATE_PATH/LIVE_STEP and RESUME_PATH/RESUME_STEP
# (each a path + step, or None).
import hashlib, json, zipfile

REQUIRED_STATE_FILES = (
    "adapter_config.json", "optimizer.pt", "scheduler.pt",
    "rng_state.pth", "trainer_state.json",
)
ADAPTER_WEIGHT_NAMES = ("adapter_model.safetensors", "adapter_model.bin")


def _sha256(p):
    h = hashlib.sha256()
    with p.open("rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def _verify_state_dir(state_dir, marker, kind):
    """Verify a state dir against its marker. Returns the step. Raises on
    any mismatch — a bundle that fails verification is never used."""
    missing = [f for f in REQUIRED_STATE_FILES if not (state_dir / f).is_file()]
    assert not missing, f"{kind} state dir {state_dir} is INCOMPLETE — missing: {missing}"
    assert any((state_dir / n).is_file() for n in ADAPTER_WEIGHT_NAMES), (
        f"{kind} state dir {state_dir} has no adapter weights")
    ts = json.loads((state_dir / "trainer_state.json").read_text())
    step = ts.get("global_step")
    assert isinstance(step, int) and step >= 0, (
        f"{kind} trainer_state.json has no valid global_step (got {step!r})")
    assert step == marker["step"], (
        f"{kind} marker step {marker['step']} disagrees with "
        f"trainer_state.json global_step {step}")
    assert marker.get("run_id"), f"{kind} marker has no run_id"
    for name, want in marker.get("files", {}).items():
        got = _sha256(state_dir / name)
        assert got == want, f"{kind} file {name} hash MISMATCH — bundle is corrupt"
    return step


def _find_bundles(marker_name, zip_prefix):
    """Find state bundles under /kaggle/input: an extracted dir holding the
    marker, or a zip whose name starts with the prefix. Returns a list of
    (marker_path, state_dir) pairs, state_dir already extracted."""
    found = []
    inp = Path("/kaggle/input")
    for mpath in sorted(inp.rglob(marker_name)):
        found.append((mpath, mpath.parent))
    for zpath in sorted(inp.rglob(f"{zip_prefix}*.zip")):
        dest = Path("/kaggle/working") / zpath.stem
        if not dest.exists():
            with zipfile.ZipFile(zpath) as z:
                z.extractall(dest)
            print("extracted", zpath, "->", dest)
        for mpath in sorted(dest.rglob(marker_name)):
            found.append((mpath, mpath.parent))
    return found


LIVE_STATE_PATH = None
LIVE_STEP = None
RESUME_PATH = None
RESUME_STEP = None

# --- live state (PRIMARY handoff) ---
for mpath, sdir in _find_bundles("live_state.json", "llmbuilder-live-state-"):
    marker = json.loads(mpath.read_text())
    assert marker.get("schema") == "llmbuilder-live-state/v1", (
        f"live_state.json schema mismatch: {marker.get('schema')!r}")
    step = _verify_state_dir(sdir, marker, "live-state")
    if LIVE_STATE_PATH is None or step > LIVE_STEP:
        LIVE_STATE_PATH, LIVE_STEP = sdir, step
    print(f"live-state bundle verified: {mpath} (step {step}, "
          f"run {marker.get('run_id')}, from {marker.get('source_worker')})")

# --- checkpoint (LAST RESORT only) ---
for mpath, sdir in _find_bundles("handoff.json", "llmbuilder-checkpoint-"):
    marker = json.loads(mpath.read_text())
    if marker.get("kind") != "checkpoint":
        continue
    assert marker.get("schema") == "llmbuilder-checkpoint-handoff/v1", (
        f"handoff.json schema mismatch: {marker.get('schema')!r}")
    step = _verify_state_dir(sdir, marker, "checkpoint")
    if RESUME_PATH is None or step > RESUME_STEP:
        RESUME_PATH, RESUME_STEP = sdir, step
    print(f"checkpoint bundle verified: {mpath} (step {step}, "
          f"run {marker.get('run_id')}, from {marker.get('source_worker')})")

if LIVE_STATE_PATH is not None:
    print(f"\nMODE: LIVE STATE (primary handoff) — step {LIVE_STEP}, "
          f"from {LIVE_STATE_PATH}")
    if RESUME_PATH is not None:
        print(f"(a checkpoint at step {RESUME_STEP} is also attached — "
              f"ignored: live state wins, it is fresher)")
elif RESUME_PATH is not None:
    print(f"\nMODE: CHECKPOINT (LAST RESORT) — step {RESUME_STEP}, "
          f"from {RESUME_PATH}")
    print("No live state was attached, so this is the worst-case fallback "
          "(the live hold is gone) or a cold-start worker. The checkpoint "
          "is the last resort, not the handoff.")
else:
    print("\nMODE: FRESH START — no state bundle attached (no live state, "
          "no checkpoint). Full 152-step run from step 0.")


In [ ]:
# FULL TRAINING RUN — mode-aware. ~67 min fresh on a Kaggle T4 (152 steps,
# QLoRA); a last-resort checkpoint resume runs only the remaining steps.
# Checkpoints land in /kaggle/working/run1/checkpoint-{25,50,75,100,125,150}
# plus the final adapter at /kaggle/working/run1. Output streams live below.
if LIVE_STATE_PATH is not None:
    # LIVE STATE received — GPU-death jump-in (primary handoff).
    # The CPU live holder produced this bundle: it held the live state
    # in parallel (stagnant — holds only, does not train) and served it
    # to this GPU worker. The live-state dir has exactly the checkpoint
    # content the trainer restores (adapter, optimizer, scheduler, RNG,
    # trainer_state). Verified against pinned transformers 5.22.2:
    # _load_from_checkpoint checks only for weight files in the dir (no
    # checkpoint-* name check), and TrainerState loads from
    # trainer_state.json in the dir. So resume from the live-state dir
    # via --resume-from-checkpoint, exactly as from a checkpoint dir —
    # the run continues from the live state, steps LIVE_STEP+1-152.
    # This is NOT a checkpoint resume: the state came from the CPU
    # holder, not from a disk checkpoint. (train_lora.py is NOT edited —
    # the resume path is its existing --resume-from-checkpoint.)
    print(f"LIVE STATE jump-in from step {LIVE_STEP} ({LIVE_STATE_PATH}) — running steps {LIVE_STEP + 1}-152.")
    run_cmd([sys.executable, str(TRAIN), "--dataset", str(DATA), "--config", str(CFG),
             "--output-dir", str(OUT), "--save-steps", "25",
             "--resume-from-checkpoint", str(LIVE_STATE_PATH)],
            f"LIVE STATE jump-in (steps {LIVE_STEP + 1}-152)")
elif RESUME_PATH is not None:
    # LAST RESORT (checkpoint): the live hold is gone (or this is a cold-start
    # worker that was never in the parallel run). Resume from the latest
    # checkpoint — the trainer restores adapter, optimizer, scheduler, RNG
    # and step count, and runs only the remaining steps (e.g. from step 50:
    # steps 51-152). Verified against the pinned transformers 5.22.2
    # Trainer source (_load_from_checkpoint + _load_optimizer_and_scheduler
    # + TrainerState.load_from_json + skip_first_batches).
    print(f"LAST-RESORT checkpoint resume from step {RESUME_STEP} "
          f"({RESUME_PATH}) — running steps {RESUME_STEP + 1}-152.")
    run_cmd([sys.executable, str(TRAIN), "--dataset", str(DATA), "--config", str(CFG),
             "--output-dir", str(OUT), "--save-steps", "25",
             "--resume-from-checkpoint", str(RESUME_PATH)],
            f"LAST-RESORT checkpoint resume (steps {RESUME_STEP + 1}-152)")
else:
    # Fresh start: no state attached.
    run_cmd([sys.executable, str(TRAIN), "--dataset", str(DATA), "--config", str(CFG),
             "--output-dir", str(OUT), "--save-steps", "25"], "FULL TRAINING RUN (152 steps)")


In [ ]:
# Post-run verification: checkpoints, manifest, adapter hash, download zip.
# Only the FINAL adapter + tokenizer + manifest are zipped (not the ~1.5 GB
# of checkpoints — those stay in /kaggle/working/run1 as notebook output).
import hashlib, json, zipfile
mode = ("live-state" if LIVE_STATE_PATH is not None
        else "checkpoint-last-resort" if RESUME_PATH is not None
        else "fresh")
print("run mode:", mode)
if LIVE_STEP is not None:
    print("live state received at step:", LIVE_STEP)
if RESUME_STEP is not None:
    print("last-resort checkpoint resumed from step:", RESUME_STEP)
run = Path("/kaggle/working/run1")
print("run dir contents:", sorted(p.name for p in run.iterdir()))
print("checkpoints:", sorted(p.name for p in run.glob("checkpoint-*")))
man = json.loads((run / "run_manifest.json").read_text())
print(json.dumps({"run_id": man["run_id"], "model": man["model"],
                  "dataset": man["dataset"], "config": man["config"]}, indent=2))
adapter = run / "adapter_model.safetensors"
if adapter.exists():
    print("adapter sha256:", hashlib.sha256(adapter.read_bytes()).hexdigest(),
          f"({adapter.stat().st_size / 1e6:.1f} MB)")
zpath = Path("/kaggle/working/llmbuilder-lora-run1.zip")
with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
    for p in run.iterdir():
        if p.is_file():
            z.write(p, p.name)
print("wrote", zpath, f"({zpath.stat().st_size / 1e6:.1f} MB) — download from the notebook Output tab")
